# Experiment 2 — actual pilot

Runs the approved 20 reference-training runs on Kaggle, with the two reference regimes for each of the ten environment/seed pairs. Each reference run uses 1,000,000 raw environment steps and the 15-checkpoint probe plus four sequential, isolated policy branches. This is the scientific pilot; the preceding GPU qualification was a short engineering check only.

Pinned implementation: `a7b229335b9737ab1ea1fe1b9f428bbc3ea26a6a` on `codex/exp2-online-prototype-20261107`.

Kaggle requests NVIDIA T4 GPUs and internet. The scheduler requests a maximum of two jobs per GPU, but the attached source-bound qualification establishes **one job per GPU** because same-GPU concurrency was not tested. Each run remains single-GPU. W&B logs online to the existing `Experiment-2-Pilot-Testing` project. The Kaggle `WANDB_API_KEY` secret must be enabled for this notebook. Training stops before GPU work if Kaggle cannot expose it.

Completed run states and analysis are saved under `/kaggle/working/exp2-actual-pilot`; a future Kaggle version can resume from an attached private recovery dataset with the same directory layout. A notebook timeout is not a completed scientific pilot.

In [ ]:
import os, sys, platform, subprocess, tempfile, shutil, socket, time, json, hashlib
from pathlib import Path

# Authenticate W&B before downloading or installing the research runtime.
if not os.environ.get("WANDB_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        raise RuntimeError("Kaggle could not read WANDB_API_KEY. In this notebook, enable the existing W&B secret under Add-ons > Secrets, then rerun. The key was not written into the notebook.") from None
if not os.environ.get("WANDB_API_KEY"):
    raise RuntimeError("WANDB_API_KEY is empty; no training was started.")

os.environ.update({
    "JAX_PLATFORMS": "cuda,cpu",
    "MUJOCO_GL": "egl",
    "PYOPENGL_PLATFORM": "egl",
    "EGL_PLATFORM": "surfaceless",
    "MUJOCO_EGL_DEVICE_ID": "0",
    "WANDB_MODE": "online",
    "WANDB_PROJECT": "Experiment-2-Pilot-Testing",
    "OPENBLAS_NUM_THREADS": "1",
    "OMP_NUM_THREADS": "1",
    "XLA_PYTHON_CLIENT_PREALLOCATE": "false",
})

SOURCE_SHA = "a7b229335b9737ab1ea1fe1b9f428bbc3ea26a6a"
SOURCE_BRANCH = "codex/exp2-online-prototype-20261107"
REPO = Path("/kaggle/temp/scaling-drl-research")
REPO.parent.mkdir(parents=True, exist_ok=True)

def wait_for_dns(host, attempts=6, delay=10):
    for attempt in range(attempts):
        try:
            socket.getaddrinfo(host, 443, type=socket.SOCK_STREAM)
            print("DNS available:", host)
            return
        except socket.gaierror as exc:
            print(f"DNS attempt {attempt + 1}/{attempts} failed for {host}: {exc}", flush=True)
            if attempt + 1 < attempts:
                time.sleep(delay)
    raise RuntimeError(f"Kaggle cannot resolve {host}; no training started.")

for host in ("api.wandb.ai", "github.com", "pypi.org", "files.pythonhosted.org"):
    wait_for_dns(host)

if not REPO.exists():
    for attempt in range(3):
        with tempfile.TemporaryDirectory(prefix="exp2-clone-", dir=REPO.parent) as tmp:
            destination = Path(tmp) / "source"
            result = subprocess.run(
                ["git", "clone", "--quiet", "--single-branch", "--branch", SOURCE_BRANCH,
                 "https://github.com/kaves29/scaling-drl-research.git", str(destination)],
                text=True, capture_output=True,
            )
            if result.returncode == 0:
                shutil.move(str(destination), str(REPO))
                break
            print(f"Git clone attempt {attempt + 1}/3 failed: {result.stderr.strip()}", flush=True)
        if attempt < 2:
            time.sleep(10)
    else:
        raise RuntimeError("Could not clone the pinned Experiment 2 source after three attempts.")
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_SHA], cwd=REPO, check=True)
actual_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_sha == SOURCE_SHA, f"Source mismatch: {actual_sha}"

requirements = (REPO / "main/requirements.txt").read_text()
install_dm_control_without_optional_mazes = sys.version_info >= (3, 13)
if install_dm_control_without_optional_mazes:
    replacements = {"numpy==1.26.4": "numpy==2.1.3", "pandas==2.1.4": "pandas==2.2.3", "matplotlib==3.9.0": "matplotlib==3.9.2", "Shimmy==2.0.0": "Shimmy==1.3.0"}
    for old, new in replacements.items():
        assert requirements.count(old) == 1, f"Expected exactly one {old} pin"
        requirements = requirements.replace(old, new)
    for unused_pin in ("labmaze==1.0.6", "dm_control==1.0.38"):
        assert requirements.count(unused_pin) == 1, f"Expected exactly one {unused_pin} pin"
        requirements = requirements.replace(unused_pin + "\n", "")
runtime_requirements = Path("/kaggle/temp/exp2-runtime-requirements.txt")
runtime_requirements.write_text(requirements)
print("Python runtime:", platform.python_version(), "; installing pinned runtime requirements")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(runtime_requirements)], check=True)
if install_dm_control_without_optional_mazes:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "dm_control==1.0.38"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "jax-cuda13-plugin", "jax-cuda13-pjrt"], check=True, stdout=subprocess.DEVNULL)
    print("Removed Kaggle CUDA 13 JAX plugins to prevent conflicts with the pinned CUDA 12 backend.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]==0.4.34"], check=True)

compat = Path("/kaggle/temp/exp2-python-compat")
compat.mkdir(exist_ok=True)
(compat / "sitecustomize.py").write_text("import numpy as np\nif not hasattr(np, 'issctype'):\n    np.issctype = lambda rep: isinstance(rep, type) and issubclass(rep, np.generic)\n")
os.environ["PYTHONPATH"] = str(compat) + os.pathsep + str(REPO / "main")

# Validate the real online W&B credential, entity, and requested project before GPU work.
import wandb
if not wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True):
    raise RuntimeError("W&B rejected the configured credential; no training was started.")
api = wandb.Api(timeout=30)
entity = api.default_entity
assert entity, "W&B authenticated but returned no default entity."
project = api.project("Experiment-2-Pilot-Testing", entity=entity)
assert project is not None, "The requested Experiment-2-Pilot-Testing project is not visible to this W&B identity."
os.environ["WANDB_ENTITY"] = entity
print("W&B authentication and requested project verified; online logging enabled.")

In [ ]:
import json, shutil, subprocess, sys, hashlib
from pathlib import Path

OUTPUT = Path("/kaggle/working/exp2-actual-pilot")
OUTPUT.mkdir(parents=True, exist_ok=True)

# On later sessions, restore the previous Kaggle output from the private recovery
# dataset. The first run has no recovery dataset attached.
resume_candidates = list(Path("/kaggle/input").glob("*/exp2-actual-pilot"))
if resume_candidates:
    if len(resume_candidates) != 1:
        raise RuntimeError(f"Expected one pilot recovery directory; found {len(resume_candidates)}")
    source = resume_candidates[0]
    for item in source.iterdir():
        target = OUTPUT / item.name
        if target.exists():
            continue
        if item.is_dir():
            shutil.copytree(item, target)
        else:
            shutil.copy2(item, target)
    print("Restored pilot state from the attached recovery dataset.")

SOURCE_SHA = "a7b229335b9737ab1ea1fe1b9f428bbc3ea26a6a"
QUALIFICATION_SHA256 = "e42829e20049bfa75545f96cafc42d57184f85c2e4345171468b02036db63d7a"
# Verified receipt downloaded from the successful Kaggle GPU qualification.
qualification = {
    "source_commit": SOURCE_SHA,
    "source_sha256": "08ea40601c34e3b08a1d2e67fd1ab66bb965dd02478e6ba42be47a7fbc676ab9",
    "full_branch_horizon": True,
    "plasticity_probe": True,
    "restart": True,
    "concurrent_pass": False,
    "qualification_receipt_sha256": QUALIFICATION_SHA256,
    "reference_raw_steps": 10000,
    "branch_raw_steps": 10000,
    "jobs_per_gpu": 1,
    "devices": {},
}
assert qualification["source_commit"] == SOURCE_SHA
assert all(qualification[k] is True for k in ("full_branch_horizon", "plasticity_probe", "restart"))
(OUTPUT / "qualification-evidence.json").write_text(json.dumps(qualification, indent=2))

inventory = subprocess.check_output([
    "nvidia-smi", "--query-gpu=index,uuid,name,memory.total,memory.free",
    "--format=csv,noheader,nounits",
], text=True)
devices = [line.strip() for line in inventory.splitlines() if line.strip()]
assert devices, "No NVIDIA GPU detected; pilot was not started."
print("Actual GPU inventory:", devices)

manifest = {
    "status": "launching",
    "source_commit": SOURCE_SHA,
    "qualification_receipt_sha256": QUALIFICATION_SHA256,
    "pilot_population": 20,
    "reference_raw_steps_each": 1000000,
    "checkpoints_each": 15,
    "branch_raw_steps_each": 10000,
    "requested_jobs_per_gpu": 2,
    "qualified_jobs_per_gpu": 1,
    "wandb_mode": "online",
    "wandb_project": "Experiment-2-Pilot-Testing",
    "devices": devices,
    "resume_dataset_attached": bool(resume_candidates),
}
(OUTPUT / "pilot-launch-manifest.json").write_text(json.dumps(manifest, indent=2))

run_env = dict(os.environ, XLA_PYTHON_CLIENT_PREALLOCATE="false")
command = [
    sys.executable, "-m", "experiments.exp12.revised_exp2_scheduler",
    "--output", str(OUTPUT / "pilot"),
    "--jobs-per-gpu", "2",
    "--qualification", str(OUTPUT / "qualification-evidence.json"),
]
print("Starting the 20-run scientific pilot; scheduler will admit at most one run per GPU.", flush=True)
with (OUTPUT / "scheduler.log").open("a", buffering=1) as log:
    subprocess.run(command, cwd="/kaggle/temp/scaling-drl-research/main", env=run_env,
                   stdout=log, stderr=subprocess.STDOUT, check=True)
manifest["status"] = "complete"
(OUTPUT / "pilot-launch-manifest.json").write_text(json.dumps(manifest, indent=2))
print("All 20 reference runs completed. Results and resumable states:", OUTPUT / "pilot")